In [1]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

!pip install -U bitsandbytes

import gc, math, torch, time
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.5 MB/s eta 0:00:00


In [2]:
# IDs
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
finetuned_repo_id_4bit = "eduhuemar001/tinyllama-german-sentiment-4bit"
finetuned_repo_id_8bit = "eduhuemar001/tinyllama-german-sentiment-8bit"

# Helpers
def print_gpu_mem(tag=""):
    if not torch.cuda.is_available():
        print(f"[{tag}] CUDA not available"); return 0
    torch.cuda.synchronize()
    a = torch.cuda.memory_allocated()
    print(f"[{tag}] allocated={a/1e9:.3f} GB"); return a

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

# ---------- fp16 base (reference) ----------
free_gpu()
print_gpu_mem("before fp16")
base_fp16 = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="cuda" if torch.cuda.is_available() else None,
)
base_fp16.eval()
alloc_fp16 = print_gpu_mem("after  fp16")
del base_fp16; free_gpu(); print_gpu_mem("after unload fp16")

# ---------- 4-bit: load the *fine-tuned* model ----------
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16
quant_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print_gpu_mem("before 4-bit finetuned")
try:
    base_4bit = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=quant_config_4bit, device_map="auto",
    )
    model_4bit = PeftModel.from_pretrained(base_4bit, finetuned_repo_id_4bit)
except Exception:
    model_4bit = AutoModelForCausalLM.from_pretrained(
        finetuned_repo_id_4bit, quantization_config=quant_config_4bit, device_map="auto",
    )
model_4bit.eval(); model_4bit.config.use_cache = True
alloc_4bit = print_gpu_mem("after  4-bit finetuned")

# cleanup before 8-bit section (to measure fairly)
del model_4bit
free_gpu()
print_gpu_mem("after unload 4-bit")

# ---------- 8-bit: load the *fine-tuned* model ----------
quant_config_8bit = BitsAndBytesConfig(load_in_8bit=True)

print_gpu_mem("before 8-bit finetuned")
try:
    base_8bit = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=quant_config_8bit, device_map="auto",
    )
    model_8bit = PeftModel.from_pretrained(base_8bit, finetuned_repo_id_8bit)
except Exception:
    model_8bit = AutoModelForCausalLM.from_pretrained(
        finetuned_repo_id_8bit, quantization_config=quant_config_8bit, device_map="auto",
    )
model_8bit.eval(); model_8bit.config.use_cache = True
alloc_8bit = print_gpu_mem("after  8-bit finetuned")

print("\n=== GPU allocated deltas ===")
print(f"fp16 base        ≈ {alloc_fp16/1e9:.3f} GB")
print(f"4-bit finetuned  ≈ {alloc_4bit/1e9:.3f} GB")
print(f"8-bit finetuned  ≈ {alloc_8bit/1e9:.3f} GB")

[before fp16] allocated=0.000 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/9.02M [00:00<?, ?B/s]

[after  4-bit finetuned] allocated=0.781 GB
[after unload 4-bit] allocated=0.781 GB
[before 8-bit finetuned] allocated=0.781 GB


adapter_config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

[after  8-bit finetuned] allocated=2.028 GB

=== GPU allocated deltas ===
fp16 base        ≈ 2.200 GB
4-bit finetuned  ≈ 0.781 GB
8-bit finetuned  ≈ 2.028 GB
